<a href="https://colab.research.google.com/github/Aleeza-Nadeem/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aleeza-Nadeem/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This notebook turns my validated Week-5/6 model into a human-reviewed content action playbook (CTR / engagement opportunity lane). It is a planning aid, non-production, and every number is observed / measured / directional, for decision support only.

In [1]:
import pandas as pd, numpy as np, json, os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

url = "https://huggingface.co/datasets/FlyRank/internship-starter/resolve/main/content_refresh_anonymized.csv"
df = pd.read_csv(url)
need = ["clicks_last_30d","impressions_prev_30d","clicks_prev_30d","sessions_prev_30d","avg_position","client_id"]
d = df.dropna(subset=need).copy()
d = d[d.impressions_prev_30d > 0].reset_index(drop=True)

num = ["impressions_prev_30d","clicks_prev_30d","sessions_prev_30d","search_volume","competition","cpc",
       "word_count","content_age_days","days_since_last_update","avg_position"]
cat = ["content_type","main_intent"]
X = d[num].copy()
for c_ in num: X[c_] = X[c_].fillna(X[c_].median())
X = pd.concat([X, pd.get_dummies(d[cat].fillna("unknown"), dtype=float)], axis=1)
print("pages:", len(d), "| clients:", d.client_id.nunique())

pages: 26612 | clients: 31


## 1. Ranked actions + reason codes

**How the queue is built (plain words):**
1. A Random Forest estimates, for every page, the chance it lands in the top 20% for next-30-day clicks. Scores come from **out-of-fold, client-grouped** predictions, so no page is scored by a model that saw its own client.
2. The Week-4 rule estimates the clicks a page leaves on the table (impressions x gap to the typical CTR for its position), from the previous 30 days.
3. **Priority = clicks left on the table x model chance.** Big, visible, under-clicked pages the model also rates highly come first.
4. Each page gets one **archetype**, which gives one **reason code** and one **action**:

| Reason code | Archetype (plain words) | Action |
|---|---|---|
| `UNDERCLICKED_TOP10` | Position 10 or better, CTR below typical for that position | Rewrite title and meta description |
| `PAGE2_CLIMBER` | Position 10 to 20 | Improve content and internal links, then monitor |
| `HEALTHY_TOP10` | Position 10 or better, CTR at or above typical | Protect: no change, monitor |
| `LOW_VISIBILITY` | Position beyond 20 | Review for merge or prune; do not rewrite titles |

In [2]:
gkf = GroupKFold(n_splits=5)
prob = np.zeros(len(d)); aucs = []
for tr, te in gkf.split(X, groups=d.client_id):
    cut = d.loc[tr, "clicks_last_30d"].quantile(0.80)
    y_tr = (d.clicks_last_30d.iloc[tr] >= cut).astype(int)
    rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=20, n_jobs=-1, random_state=42, class_weight="balanced")
    rf.fit(X.iloc[tr], y_tr)
    prob[te] = rf.predict_proba(X.iloc[te])[:,1]
    aucs.append(roc_auc_score((d.clicks_last_30d.iloc[te] >= cut).astype(int), prob[te]))
d["model_prob"] = prob
print("out-of-fold AUC by client-group fold:", np.round(aucs,3))

# Week-4 opportunity, from the previous 30 days; typical CTR = median by position bucket
d["ctr_prev"] = (d.clicks_prev_30d / d.impressions_prev_30d).clip(0, 1)
d["pos_bucket"] = pd.cut(d.avg_position, [0,3,5,10,20,1000])
d["expected_ctr"] = d.groupby("pos_bucket", observed=True)["ctr_prev"].transform("median").astype(float)
d["clicks_left"] = (d.impressions_prev_30d * (d.expected_ctr - d.ctr_prev)).clip(lower=0).fillna(0)

def archetype(r):
    if r.avg_position <= 10:
        return "UNDERCLICKED_TOP10" if r.ctr_prev < r.expected_ctr else "HEALTHY_TOP10"
    if r.avg_position <= 20: return "PAGE2_CLIMBER"
    return "LOW_VISIBILITY"
action_map = {"UNDERCLICKED_TOP10":"Rewrite title and meta description",
              "PAGE2_CLIMBER":"Improve content and internal links, then monitor",
              "HEALTHY_TOP10":"Protect: no change, monitor",
              "LOW_VISIBILITY":"Review for merge or prune"}
d["reason_code"] = d.apply(archetype, axis=1)
d["action"] = d.reason_code.map(action_map)
d["priority"] = d.clicks_left * d.model_prob
queue = d[d.reason_code.isin(["UNDERCLICKED_TOP10","PAGE2_CLIMBER"]) & (d.priority > 0)].sort_values("priority", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1
print("archetype counts:"); print(d.reason_code.value_counts())
queue[["rank","reason_code","action","priority","avg_position","ctr_prev","model_prob"]].head(10)   # ids stay out of the displayed output

out-of-fold AUC by client-group fold: [0.965 0.945 0.963 0.97  0.986]
archetype counts:
reason_code
LOW_VISIBILITY        8075
PAGE2_CLIMBER         6958
HEALTHY_TOP10         6200
UNDERCLICKED_TOP10    5379
Name: count, dtype: int64


,rank,reason_code,action,priority,avg_position,ctr_prev,model_prob
0,1,UNDERCLICKED_TOP10,Rewrite title and meta description,199.758591,4.2,0.001143,0.979627
1,2,UNDERCLICKED_TOP10,Rewrite title and meta description,117.144764,4.7,0.000864,0.995623
2,3,UNDERCLICKED_TOP10,Rewrite title and meta description,54.666990,4.8,0.000630,0.993758
3,4,UNDERCLICKED_TOP10,Rewrite title and meta description,52.208202,4.9,0.000461,0.994010
4,5,UNDERCLICKED_TOP10,Rewrite title and meta description,51.196143,4.0,0.000765,0.997556
5,6,UNDERCLICKED_TOP10,Rewrite title and meta description,48.785568,4.5,0.000753,0.992682
6,7,UNDERCLICKED_TOP10,Rewrite title and meta description,38.371551,3.7,0.001024,0.984043
7,8,UNDERCLICKED_TOP10,Rewrite title and meta description,36.442492,3.4,0.000440,0.981794
8,9,UNDERCLICKED_TOP10,Rewrite title and meta description,36.020080,7.8,0.000224,0.976653
9,10,UNDERCLICKED_TOP10,Rewrite title and meta description,32.501339,4.7,0.001025,0.991183


**Decay / refresh insight.** The table below looks at how recent page change differs by how long ago a page was last updated. It is descriptive only: it shows association in this sample, not that updating a page causes a change.

In [3]:
if "trend_pct" in df.columns:
    z = df.dropna(subset=["days_since_last_update","trend_pct"]).copy()
    z["since_update"] = pd.cut(z.days_since_last_update, [0,20,60,120,400])
    decay = z.groupby("since_update", observed=True).agg(n=("trend_pct","size"),
            median_trend_pct=("trend_pct","median"), share_declining=("trend_pct", lambda s: (s<0).mean())).round(3)
    print(decay)
else:
    decay = None

                  n  median_trend_pct  share_declining
since_update                                          
(0, 20]       12992            -41.40            0.759
(20, 60]       4590            -13.50            0.611
(60, 120]      8868            -34.40            0.783
(120, 400]      162            -45.95            0.667


**Decay insight:** Staleness shows no clean pattern. The median recent trend is -41.4% for pages updated within the last 20 days, -13.5% for 20 to 60 days, -34.4% for 60 to 120 days and -46.0% for 120 to 400 days (only 162 pages). Most pages are declining in every group, so this matches my Week-4 finding that staleness is a mixed signal. This is observed association only and does not show that a refresh causes recovery.

## 2. Intended use and limits

**Who uses this:** a content or SEO team deciding which pages to look at first when time is limited.
**For what:** a starting shortlist of pages whose click-through rate looks low for where they rank, with a suggested first action. It is decision-support for human review, not an instruction.
**Where it stops being valid:**
- It was built and checked on a small anonymized sample (about 30k pages); results on other sites or the full warehouse may differ.
- The model is a ranking aid for the next 30 days based on the previous 30 days; it does not predict beyond that.
- In Weeks 5 and 6, simply ranking pages by last month's clicks did about as well as the Random Forest on held-out clients, so the model's chance mostly reflects page size and persistence. The queue's priority therefore leans on the clicks-left rule; treat the model chance as a mild re-weighting, not independent evidence.
- The "typical CTR" per position bucket (median) and the median fills for missing values are computed on all pages in this sample, which is acceptable for a planning queue but is not a held-out evaluation.
- `avg_position` is a 90-day field that overlaps the last 30 days, so a mild window overlap remains.
- CTR can be low for reasons a title cannot fix: search features, ads, AI answers, branded searches, or a tracking issue.
- The data shows association only. It does not show that a rewrite or refresh causes more clicks, and it says nothing about how Google works.

**Cost / value thinking.** The queue's estimated "clicks left on the table" is a ceiling, not a forecast. The table below shows how much of that ceiling the first N pages hold, so a team can choose how many pages are worth the effort.

The review-time figure is an **assumption** (30 minutes per page to check the data and draft a new title and description); replace it with the team's real number. "Clicks left per review hour" is a ceiling on 30-day clicks, not a forecast.


In [4]:
MIN_PER_PAGE = 30   # ASSUMPTION: minutes for a person to review one page and draft a new title/meta; edit to your team's real figure
q = queue[queue.reason_code == "UNDERCLICKED_TOP10"].copy()
total = q.clicks_left.sum()
rows = []
for n in [10, 25, 50, 100, 250]:
    if n <= len(q):
        got = q.clicks_left.head(n).sum()
        hours = n * MIN_PER_PAGE / 60
        rows.append({"top_n_pages": n,
                     "share_of_estimated_clicks_left": round(got / total, 3),
                     "assumed_review_hours": round(hours, 1),
                     "clicks_left_ceiling_per_review_hour": round(got / hours, 1)})
value_table = pd.DataFrame(rows)
print("pages in UNDERCLICKED_TOP10 queue:", len(q))
value_table


pages in UNDERCLICKED_TOP10 queue: 5378


,top_n_pages,share_of_estimated_clicks_left,assumed_review_hours,clicks_left_ceiling_per_review_hour
0,10,0.193,5.0,135.1
1,25,0.308,12.5,86.0
2,50,0.402,25.0,56.1
3,100,0.522,50.0,36.5
4,250,0.687,125.0,19.2


## 3. Human review + the no-go list

**A person must check before acting on any row:**
1. Open the page's real search data for the last 90 days and confirm the CTR and position are not a tracking or data glitch (a CTR near zero on very high impressions is the first thing to doubt).
2. Check whether the searches are branded or navigational, where low CTR is normal.
3. Look at the results page: if a snippet, ad or AI answer takes the clicks, a new title may not help.
4. Read the current title and meta description and decide whether a rewrite is genuinely needed.
5. Record the date of the change, so any later movement can be reviewed fairly (and not assumed to be caused by the change).

**Never automate:**
- Publishing or rewriting titles, meta descriptions or page content without a person approving them.
- Deleting, merging or redirecting pages from `LOW_VISIBILITY` (prune and merge are review-only).
- Reporting results as proof that a refresh worked, or as claims about how Google ranks pages.
- Using the queue on a new client or site without re-checking the data and the model first.
- Sharing client names, URLs, queries or raw exports.

## 4. Monitoring / retrain triggers

Light-touch checks that tell me the recommendations may have gone stale:
- **Accuracy drift:** the five client-group folds scored ROC AUC 0.945 to 0.986 (recorded in the metrics file below). If ROC AUC on newly arrived pages falls below about 0.90 (the lowest fold minus 0.05), review the model before using the queue.
- **CTR benchmark drift:** if the median CTR for any position bucket moves by more than about a quarter from the values recorded below, rebuild the "typical CTR" table.
- **Data shape change:** a new data release, new client mix, or new columns means re-running the leakage audit.
- **Review outcomes:** if reviewers reject many rows for the same reason (for example branded searches), add a rule or a feature for it.
- **Schedule:** re-check at least once per quarter, even if nothing else changed.

## 5. Exports for the paper

Writes the ranked queue (stays out of git, regenerated each run), two figures to reuse, and the metrics JSON (commit the figure and the JSON).

In [5]:
os.makedirs("work/outputs", exist_ok=True); os.makedirs("work/figures", exist_ok=True)
queue[["rank","content_id","reason_code","action","priority"]].to_csv("work/outputs/w07_action_queue.csv", index=False)

counts = d.reason_code.value_counts().reindex(["UNDERCLICKED_TOP10","PAGE2_CLIMBER","HEALTHY_TOP10","LOW_VISIBILITY"])
fig, ax = plt.subplots(figsize=(7,3.6))
ax.barh(counts.index[::-1], counts.values[::-1], color="#4c78a8")
ax.set_xlabel("Number of pages"); ax.set_title("Pages by archetype / reason code (observed, directional)")
plt.tight_layout(); plt.savefig("work/figures/w07_actions_by_archetype.png", dpi=150); plt.close()

# second reusable figure: how fast the estimated clicks-left ceiling is captured as pages are reviewed
cum = (q.clicks_left.cumsum() / q.clicks_left.sum()).values
fig, ax = plt.subplots(figsize=(7,3.6))
ax.plot(np.arange(1, len(cum)+1), cum, color="#4c78a8")
ax.set_xscale("log"); ax.set_xlabel("Pages reviewed, in queue order (log scale)")
ax.set_ylabel("Share of estimated clicks left"); ax.set_ylim(0, 1.02)
ax.set_title("Share of the estimated ceiling held by the first N pages (observed, directional)")
plt.tight_layout(); plt.savefig("work/figures/w07_clicks_left_curve.png", dpi=150); plt.close()

typical = d.groupby("pos_bucket", observed=True)["ctr_prev"].median().round(4)
json.dump({"queue_rows": int(len(queue)),
           "reason_code_counts": {k:int(v) for k,v in counts.items()},
           "out_of_fold_auc_by_group_fold": [round(a,3) for a in aucs],
           "typical_ctr_by_position_bucket": {str(k): float(v) for k,v in typical.items()},
           "cost_value": value_table.to_dict(orient="records"),
           "cost_value_assumption": f"{MIN_PER_PAGE} minutes review time per page (assumed, edit to real figure)",
           "monitoring_thresholds": {"review_if_auc_below": round(min(aucs) - 0.05, 2),
                                     "review_if_ctr_bucket_median_shifts_by": 0.25},
           "note": "observed / directional / decision-support; no causal claim"},
          open("work/outputs/w07_playbook_metrics.json","w"), indent=2, default=float)
print("exported queue, 2 figures and metrics:", len(queue), "rows")

exported queue, 2 figures and metrics: 5378 rows


In [6]:
# Download the metrics JSON and the figures (Colab only). Commit them to work/outputs/ and work/figures/ in your repo.
try:
    from google.colab import files
    for f in ["work/outputs/w07_playbook_metrics.json",
              "work/figures/w07_actions_by_archetype.png",
              "work/figures/w07_clicks_left_curve.png"]:
        files.download(f)   # if the browser asks, allow multiple downloads
except ImportError:
    print("Not running in Colab - the files are already in work/outputs/ and work/figures/ on this machine.")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.